# ✏️ Lesson 5.3 — Prompt Engineering: Getting the Best from AI

**AI Crash Course for Absolute Beginners**

Prompting is the skill of communicating precisely with AI models. In this notebook:
- Compare zero-shot, one-shot, and few-shot prompting
- Use Chain-of-Thought (CoT) prompting on reasoning tasks
- Apply the ROLE + TASK + FORMAT + CONSTRAINTS template
- Measure prompt quality with automated scoring
- Explore system prompts for consistent AI personas

> **Note:** Most cells use free local models. OpenAI cells are clearly marked and optional.

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install transformers torch --quiet

from transformers import pipeline
import textwrap

# We'll use a small model for demonstrations
# (swap for gpt-4o-mini via API for production quality)
generator = pipeline("text-generation", model="gpt2-medium", device=-1,
                     pad_token_id=50256)

def generate(prompt, max_new_tokens=120, temperature=0.7):
    result = generator(prompt, max_new_tokens=max_new_tokens,
                       do_sample=True, temperature=temperature,
                       num_return_sequences=1)
    return result[0]["generated_text"][len(prompt):].strip()

print("Model loaded.")

---
## Part 1 — Zero-Shot vs Few-Shot Prompting

In [ ]:
# Zero-shot: no examples provided
zero_shot_prompt = "Classify the sentiment of this review (Positive/Negative):\nReview: The movie was an absolute masterpiece with stunning visuals.\nSentiment:"

# Few-shot: examples show the pattern
few_shot_prompt = """Classify the sentiment as Positive or Negative.

Review: I loved every minute of it!
Sentiment: Positive

Review: Terrible experience, would not recommend.
Sentiment: Negative

Review: The movie was an absolute masterpiece with stunning visuals.
Sentiment:"""

print("=== Zero-shot prompt ===")
print(zero_shot_prompt)
print("\nModel output:", generate(zero_shot_prompt, max_new_tokens=10, temperature=0.1))

print("\n" + "="*50)
print("=== Few-shot prompt ===")
print(few_shot_prompt)
print("\nModel output:", generate(few_shot_prompt, max_new_tokens=10, temperature=0.1))

---
## Part 2 — Chain-of-Thought Prompting

In [ ]:
# Without CoT
direct_prompt = "Roger has 5 tennis balls. He buys 2 more cans of 3 tennis balls each. How many tennis balls does he have now? Answer:"

# With Chain-of-Thought
cot_prompt = """Solve these problems step by step.

Problem: If Sam has 4 apples and buys 3 bags of 2 apples each, how many does he have?
Reasoning: Sam starts with 4 apples. He buys 3 bags x 2 apples = 6 apples. Total = 4 + 6 = 10.
Answer: 10

Problem: Roger has 5 tennis balls. He buys 2 more cans of 3 tennis balls each. How many tennis balls does he have now?
Reasoning:"""

print("Without Chain-of-Thought:")
print("Prompt:", direct_prompt)
print("Output:", generate(direct_prompt, max_new_tokens=20, temperature=0.1))

print("\n" + "-"*50)
print("\nWith Chain-of-Thought:")
print("Prompt (last line):", "Problem: Roger... | Reasoning:")
print("Output:", generate(cot_prompt, max_new_tokens=60, temperature=0.1))

---
## Part 3 — The ROLE + TASK + FORMAT Template

In [ ]:
# Bad prompt
bad_prompt = "explain neural networks"

# Good prompt with ROLE + TASK + FORMAT + CONSTRAINTS
good_prompt = """You are an expert AI educator who teaches complete beginners.

Task: Explain how neural networks learn in 3 bullet points.
Format: Each point starts with a bold heading, followed by 1 sentence of plain English.
Constraints: No jargon. Audience is a 14-year-old. Max 80 words total.

Explanation:"""

print("BAD PROMPT:", bad_prompt)
print("Output:", generate(bad_prompt, max_new_tokens=60, temperature=0.7))

print("\n" + "="*50 + "\n")

print("GOOD PROMPT (ROLE+TASK+FORMAT+CONSTRAINTS):")
print(good_prompt)
print("Output:", generate(good_prompt, max_new_tokens=100, temperature=0.7))

---
## Part 4 — Prompt A/B Testing

In [ ]:
# Test 3 prompt variations for a creative writing task
task_topic = "climate change"

prompts = {
    "Vague":
        f"Write about {task_topic}.",

    "Specific":
        f"Write a 3-sentence opening paragraph for a news article about {task_topic} aimed at teenagers.",

    "Role + Specific + Format":
        f"""You are a Pulitzer Prize-winning journalist.
Write a 3-sentence opening for a teen magazine article about {task_topic}.
The first sentence must be a shocking statistic. End with a call to action."""
}

for name, prompt in prompts.items():
    print(f"=== {name} ===")
    print(f"Prompt: {prompt[:100]}..." if len(prompt)>100 else f"Prompt: {prompt}")
    output = generate(prompt, max_new_tokens=80, temperature=0.7)
    print(f"Output: {output}")
    print()

---
## Part 5 — Prompt Anti-Patterns to Avoid

In [ ]:
anti_patterns = {
    "Vague task":
        {"bad":  "Help me with my email.",
         "good": "Rewrite this email to be 20% shorter and more direct. Keep a professional tone. Email: [PASTE EMAIL HERE]"},

    "Leading question":
        {"bad":  "Isn't GPT-4 clearly the best AI model available?",
         "good": "Compare the top 3 LLMs available in 2025. List pros, cons, and best use cases for each."},

    "No format":
        {"bad":  "Tell me about neural network architectures.",
         "good": "List the 5 most important neural network architectures in a Markdown table with columns: Name | Best For | Year Introduced."},

    "Asking too many things":
        {"bad":  "Explain CNNs, RNNs, transformers, attention, and BERT and compare them all.",
         "good": "Explain CNNs in 3 sentences. Focus only on how convolutional filters work."},
}

for pattern, examples in anti_patterns.items():
    print(f"Anti-pattern: {pattern}")
    print(f"  BAD : {examples['bad']}")
    print(f"  GOOD: {examples['good']}")
    print()

---
## Part 6 -- (Optional) OpenAI API: System Prompts

In [ ]:
# Uncomment to run with your OpenAI API key

# import openai
# client = openai.OpenAI(api_key="sk-...")
#
# # System prompts create a persistent persona
# system_prompt = """You are Aria, a friendly AI tutor for a beginner AI course.
# Rules:
# - Always use a Ghanaian English tone — warm and encouraging
# - Use analogies from everyday life (cooking, football, market shopping)
# - Never use jargon without immediately explaining it
# - End every answer with one follow-up question to check understanding"""
#
# def chat_with_aria(user_message, history=[]):
#     history.append({"role": "user", "content": user_message})
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "system", "content": system_prompt}] + history
#     )
#     reply = response.choices[0].message.content
#     history.append({"role": "assistant", "content": reply})
#     return reply
#
# print(chat_with_aria("What is gradient descent?"))

print("OpenAI cells are commented out. Fill in your API key to run.")

---
## Prompt Engineering Cheat Sheet

| Technique | When to use | Example |
|---|---|---|
| Zero-shot | Simple, clear tasks | "Translate to French: Hello" |
| Few-shot | Format needs examples | Show 3 examples, then ask |
| Chain-of-Thought | Multi-step reasoning | "Think step by step" |
| Role prompting | Persona consistency | "You are an expert in..." |
| Self-consistency | Important decisions | Generate 5 answers, take majority |
| Decomposition | Complex tasks | Break into sub-problems |

---
## Challenge Exercises

1. **Persona A/B test**: Write the same question twice — once with "You are a pirate explaining" and once without. Compare outputs.
2. **Format control**: Ask for a JSON output with specific keys. Does the model respect the format?
3. **CoT on maths**: Try `"Let's think step by step"` on a tricky word problem. Does it improve accuracy?

---
**Next lesson:** [Lesson 5.4 — Computer Vision in Practice](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-5.4-computer-vision.ipynb)